# 🎯 Your first classifier: learn from known answers

**Concept:** supervised machine learning. **Tool:** scikit-learn (`sklearn`).

Run each code cell from top to bottom using the play triangle in Colab. This lesson uses fictional data and needs no API key. A teacher can explain the code while you focus on the questions.

[Back to the learning hub](https://github.com/bforbesc/Data-AI-with-Forbes)

**Question:** can measurements help predict whether a fictional fruit is an apple or a pear? **Features** are the measurements. A **label** is the known answer used during learning. A **classifier** predicts a category.


In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

rng = random.Random(42)
rows = []
for label, roundness_range in [("apple", (0.72, 0.95)), ("pear", (0.45, 0.80))]:
    for _ in range(60):
        rows.append({"weight_g": rng.randint(100, 220),
                     "roundness": round(rng.uniform(*roundness_range), 2),
                     "fruit": label})
data = pd.DataFrame(rows)
# These numbers were invented to contain a pattern, not measured from real fruit.
print(data.groupby("fruit").size())
data.head()


## 1. Keep some answers out of the lesson

**Training data** teaches the model. **Test data** checks it on examples it did not train on. The label column is the answer, so we keep it separate from the input features. An answer accidentally included as an input is a form of **data leakage**.


In [ ]:
features = data[["weight_g", "roundness"]]
labels = data["fruit"]
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.30, random_state=42, stratify=labels
)
print("Training examples:", len(X_train))
print("Test examples:", len(X_test))


## 2. Learn a small decision tree

A **decision tree** learns a sequence of questions about the measurements. Limiting its depth keeps that sequence short. Compare it with a **baseline** that always predicts a most frequent training label without looking at the measurements.


In [ ]:
MAX_DEPTH = 3  # Your turn later: try 1, then rerun this cell and the next.
model = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=42)
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)


## 3. Check on the held-out examples

**Accuracy** is the fraction of predictions matching the known labels. A **confusion matrix** counts correct and incorrect predictions for each category; rows are known answers and columns are predictions. A single score can hide which mistakes happened.


In [ ]:
predictions = model.predict(X_test)
print(f"Baseline test accuracy: {accuracy_score(y_test, baseline.predict(X_test)):.1%}")
print(f"Tree test accuracy: {accuracy_score(y_test, predictions):.1%}")
print("\nKnown answers (rows) versus predictions (columns):")
print(pd.DataFrame(confusion_matrix(y_test, predictions, labels=["apple", "pear"]),
                   index=["actual apple", "actual pear"],
                   columns=["predicted apple", "predicted pear"]))


## 4. Predict a new example

We do not supply a label for the new fruit. The prediction is a learned guess, not a measured fact.


In [ ]:
new_fruit = pd.DataFrame([{"weight_g": 160, "roundness": 0.85}])
print("Prediction:", model.predict(new_fruit)[0])


## Your turn: explain what changed

1. Find `MAX_DEPTH = 3`, change only the number to `1`, and rerun the training and evaluation cells. Compare against the same baseline. Did a simpler tree change the errors?
2. Why would reporting performance on the training examples alone be misleading?
3. Could this model be trusted on photos of real fruit? Explain what data and evaluation are missing.

**Check your reasoning:** there are **84 training examples** and **36 test examples**. The balanced test set makes the constant-label baseline **50% accurate**. The tree score is an observation from this invented example, not evidence of real-world fruit recognition. We supplied measurements rather than photos.

Repeatedly changing a model to improve this same test score makes the test less independent. In a larger project, use validation data for those choices and a separate final test set.

**Compare:** in [the clustering case study](https://github.com/bforbesc/clustering-web-app), no labels are supplied. That is unsupervised learning: the output is a grouping to investigate rather than a predictor trained on known answers.
